# Quick Start

`treem` is a tool for working with neuron reconstructions saved as SWC files. An SWC file stores the shape of a neuron as points and connections.

In this tutorial, you will start with one sample SWC file and follow the usual workflow: check it, clean it if needed, view it, measure it, scale it, and compare the result.

Open this notebook from the `tutorials/` folder. The files created during the tutorial will stay inside `data/` and `output/`.

## Workflow

```text
sample SWC file
      |
      v
check the file
      |
      v
convert it if needed
      |
      v
view and measure it
      |
      v
scale the reconstruction
      |
      v
compare before and after
```

## Reading the commands

Most commands in this tutorial start with `swc`, the `treem` command for SWC files.

A command usually has this structure:

```text
swc action input-file options
```

For example:

```text
swc convert data/pass_nmo_1.swc -o data/inp.swc -q
```

| Part | Meaning |
| --- | --- |
| `swc` | use the `treem` SWC command |
| `convert` | choose what to do |
| `data/pass_nmo_1.swc` | input file |
| `-o data/inp.swc` | save the result here |
| `-q` | quiet mode, used when less text output is needed |

Other actions used below are `check`, `view`, `measure`, and `modify`.

## Command helper

Run this cell once before the tutorial commands. It creates `shell_cmd()`, a small notebook helper that runs terminal commands, shows their output, and displays images when a command creates one.

In [ ]:
import shlex
import shutil
import subprocess
from pathlib import Path
from types import SimpleNamespace

from IPython.display import Image, display


def _usable_bash():
    bash_path = shutil.which("bash")
    if bash_path is None:
        return None
    probe = subprocess.run(
        [bash_path, "-lc", "command -v swc"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return bash_path if probe.returncode == 0 else None


def _portable_command(command):
    parts = shlex.split(command)
    if not parts:
        return None

    if parts[0] == "mkdir":
        folders = [part for part in parts[1:] if not part.startswith("-")]
        lines = []
        for folder in folders:
            path = Path(folder)
            existed = path.exists()
            path.mkdir(parents=True, exist_ok=True)
            if not existed:
                lines.append(f"created directory: {folder}")
        return SimpleNamespace(stdout="\n".join(lines), returncode=0)

    if parts[0] == "cp":
        verbose = "-v" in parts[1:]
        paths = [part for part in parts[1:] if part != "-v"]
        if len(paths) < 2:
            return None
        destination = Path(paths[-1])
        sources = [Path(path) for path in paths[:-1]]
        lines = []
        for source in sources:
            target = destination / source.name if destination.exists() and destination.is_dir() else destination
            shutil.copy2(source, target)
            if verbose:
                lines.append(f"'{source}' -> '{target}'")
        return SimpleNamespace(stdout="\n".join(lines), returncode=0)

    return None


def shell_cmd(command, image=None):
    print("Executed command:")
    print(command)
    bash_path = _usable_bash()
    portable_result = None if bash_path is not None else _portable_command(command)

    if portable_result is not None:
        result = portable_result
    elif bash_path is None:
        result = subprocess.run(
            command,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
    else:
        result = subprocess.run(
            [bash_path, "-lc", command],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

    output = result.stdout.rstrip()
    print("\nOutput:")
    print(output if output else "(empty)")

    if image is not None and Path(image).exists():
        display(Image(filename=image))


## Prepare working folders
Create temporary folders for input data and generated output files.

In [ ]:
shell_cmd("mkdir -pv data output")

## Get a sample reconstruction

Start with a sample SWC reconstruction from the repository test data.

In [ ]:
shell_cmd("cp -v ../tests/data/pass_nmo_1.swc data/")

## Check the file

Check the file for consistency with the SWC data format.

In [ ]:
shell_cmd("swc check data/pass_nmo_1.swc")

The sample file reports a consistency problem. Convert it to a compliant SWC file and continue with the converted file.

In [ ]:
shell_cmd("swc convert data/pass_nmo_1.swc -o data/inp.swc -q")

## View the reconstruction

Create a picture of the converted reconstruction.

In [ ]:
shell_cmd("swc view data/inp.swc -o output/inp.png", image="output/inp.png")

## Measure the reconstruction

Measure the basic morphometric features of the converted reconstruction.

In [ ]:
shell_cmd("swc measure data/inp.swc")

## Modify the reconstruction

Scale the reconstruction by a factor of 2 along the x, y, and z axes.

In [ ]:
shell_cmd("swc modify data/inp.swc -s 2 2 2 -o output/scaled.swc")

## Compare the two reconstructions

View the converted reconstruction and the scaled reconstruction together.

In [ ]:
shell_cmd(
    "swc view data/inp.swc output/scaled.swc -c cells -o output/compare.png",
    image="output/compare.png",
)

Measure both reconstructions and compare the values.

In [ ]:
shell_cmd("swc measure data/inp.swc output/scaled.swc")

Make sure that the simple scaling changed only spatial measurements and did not alter structural characteristics `breadth`, `nbranch`, `nstem`, `nterm` and `order`.